# Checking correct dtypes for columns to fix loading related issues Checking correct dtypes for columns to fix loading related issue

## Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Loading Dataset

In [8]:
file_names = ['Friday-02-03-2018_TrafficForML_CICFlowMeter.csv', 'Friday-16-02-2018_TrafficForML_CICFlowMeter.csv', 'Friday-23-02-2018_TrafficForML_CICFlowMeter.csv', 'Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv', 'Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv', 'Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv', 'Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv', 'Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv', 'Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv', 'Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv'
              ]

raw_dataset_dir = Path('../dataset/raw')

In [9]:
current_dataset_file_idx = 9
df = pd.read_csv(Path(raw_dataset_dir, file_names[current_dataset_file_idx]), low_memory=False)

In [10]:
for col in df.columns:
    print(f'{col}: ', end='')
    print(df[col].unique())

Dst Port: <StringArray>
[  '443',   '445', '49688',    '80',     '0', '49689',   '123', '49750',
 '49751', '49752',
 ...
  '8912', '39393',  '8789',  '2836', '29196',  '3174',  '1983',  '3368',
 '33758', '37777']
Length: 13651, dtype: str
Protocol: <StringArray>
['6', '0', '17', 'Protocol']
Length: 4, dtype: str
Timestamp: <StringArray>
['28/02/2018 08:22:13', '28/02/2018 08:22:15', '28/02/2018 08:22:16',
 '28/02/2018 08:22:20', '28/02/2018 08:22:21', '28/02/2018 08:22:25',
 '28/02/2018 08:22:34', '28/02/2018 08:22:23', '28/02/2018 08:22:44',
 '28/02/2018 08:22:49',
 ...
 '28/02/2018 12:47:28', '28/02/2018 04:41:36', '28/02/2018 02:58:38',
 '28/02/2018 12:32:28', '28/02/2018 05:08:19', '28/02/2018 12:55:10',
 '28/02/2018 03:24:57', '28/02/2018 12:08:22', '28/02/2018 03:07:02',
 '28/02/2018 12:52:55']
Length: 31855, dtype: str
Flow Duration: <StringArray>
[   '94658',      '206',   '165505',   '102429',      '167',   '164387',
        '0',   '131411',   '279349', '20771523',
 ...
   '11

From this output it can be seen that in each column, there is at least one value that is actually  the column name, that is to say the header row has been duplicated at least once. This observation holds for file:

1. Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
2. Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
3. Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

In [11]:
header = df.columns

header_rows = (df.astype(str) == header).all(axis=1)

print("Repeated header rows:", header_rows.sum())

Repeated header rows: 33


Repeated header count for files:

|                        File                        | Count |
|:--------------------------------------------------:|:-----:|
|  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv   |   1   |
| Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv  |  25   |
| Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv |  33   |

In [12]:
from src.core.config import config_loader

df = df.loc[~header_rows].copy()

schema_cfg = config_loader('../config/preprocessing/validation_schema.yaml')
non_numeric_cols = schema_cfg['non_numeric_col']

feature_cols = df.columns.drop(non_numeric_cols)

df[feature_cols] = df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

df.info()

<class 'pandas.DataFrame'>
Index: 613071 entries, 0 to 613103
Data columns (total 80 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Dst Port           613071 non-null  int64  
 1   Protocol           613071 non-null  int64  
 2   Timestamp          613071 non-null  str    
 3   Flow Duration      613071 non-null  int64  
 4   Tot Fwd Pkts       613071 non-null  int64  
 5   Tot Bwd Pkts       613071 non-null  int64  
 6   TotLen Fwd Pkts    613071 non-null  int64  
 7   TotLen Bwd Pkts    613071 non-null  int64  
 8   Fwd Pkt Len Max    613071 non-null  int64  
 9   Fwd Pkt Len Min    613071 non-null  int64  
 10  Fwd Pkt Len Mean   613071 non-null  float64
 11  Fwd Pkt Len Std    613071 non-null  float64
 12  Bwd Pkt Len Max    613071 non-null  int64  
 13  Bwd Pkt Len Min    613071 non-null  int64  
 14  Bwd Pkt Len Mean   613071 non-null  float64
 15  Bwd Pkt Len Std    613071 non-null  float64
 16  Flow Byts/s       